# Run reliability experiment on BKT

## Install pyBKT

In [ ]:
# Install the same pyBKT version used in the baseline

!git clone https://github.com/CAHLR/pyBKT.git
%cd pyBKT
!pip install .

Cloning into 'pyBKT'...
remote: Enumerating objects: 4655, done.
remote: Counting objects: 100% (350/350), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 4655 (delta 250), reused 292 (delta 224), pack-reused 4305 (from 1)
Receiving objects: 100% (4655/4655), 3.31 MiB | 4.40 MiB/s, done.
Resolving deltas: 100% (2224/2224), done.
/content/pyBKT
Processing /content/pyBKT
  Preparing metadata (setup.py) ... done
  Created wheel for pyBKT: filename=pyBKT-1.4.3-cp313-cp313-linux_x86_64.whl size=1130154 sha256=9db4f75254bb1d1a8de49332d668be2f1b0957386149be3c86e634090a7ef7cb
  Stored in directory: /tmp/pip-ephem-wheel-cache-_ik1hl6f/wheels/23/39/9e/901a22284c9c3d36fb209c86485f4bb3c82d33b69d8e532f2c
Successfully built pyBKT


## Imports

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os
import time
import numpy as np
import pandas as pd


RELIABILITY_DIR = (
    "/content/drive/MyDrive/"
    "education-ml-research/ASSISTments2009/"
    "reliability_experiment/"
)

# Load the same datasets used for SAKT
train_df = pd.read_csv(RELIABILITY_DIR + "train.csv")
q1_test = pd.read_csv(RELIABILITY_DIR + "q1_test.csv")
q2_test = pd.read_csv(RELIABILITY_DIR + "q2_test.csv")
q3_test = pd.read_csv(RELIABILITY_DIR + "q3_test.csv")
q4_test = pd.read_csv(RELIABILITY_DIR + "q4_test.csv")

print("Training:", train_df.shape)
print("Q1:", q1_test.shape)
print("Q2:", q2_test.shape)
print("Q3:", q3_test.shape)
print("Q4:", q4_test.shape)

Mounted at /content/drive


/tmp/ipykernel_2200/1421340122.py:18: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(RELIABILITY_DIR + "train.csv")


Training: (426580, 30)
Q1: (11444, 30)
Q2: (23524, 30)
Q3: (30184, 30)
Q4: (33802, 30)


/tmp/ipykernel_2200/1421340122.py:22: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  q4_test = pd.read_csv(RELIABILITY_DIR + "q4_test.csv")


## Prepare the data for BKT

In [ ]:
def prepare_bkt_data(data):
    data = data[
        ["order_id", "user_id", "skill_id", "correct"]
    ].copy()

    data = data.dropna(subset=["skill_id"])

    data["skill_id"] = (
        data["skill_id"]
        .astype(int)
        .astype(str)
    )

    data["correct"] = pd.to_numeric(
        data["correct"],
        errors="coerce"
    )

    data = data[
        data["correct"].isin([0, 1])
    ].copy()

    data = data.sort_values(
        ["user_id", "order_id"]
    ).reset_index(drop=True)

    return data


bkt_train = prepare_bkt_data(train_df)
bkt_q1 = prepare_bkt_data(q1_test)
bkt_q2 = prepare_bkt_data(q2_test)
bkt_q3 = prepare_bkt_data(q3_test)
bkt_q4 = prepare_bkt_data(q4_test)

print("Training interactions:", len(bkt_train))
print("Q1 interactions:", len(bkt_q1))
print("Q2 interactions:", len(bkt_q2))
print("Q3 interactions:", len(bkt_q3))
print("Q4 interactions:", len(bkt_q4))

Training interactions: 372054
Q1 interactions: 9373
Q2 interactions: 19826
Q3 interactions: 25960
Q4 interactions: 31995


## Fit the model

In [ ]:
from pyBKT.models import Model

start = time.time()

bkt_model = Model(
    seed=42,
    num_fits=1,
    parallel=True
)

bkt_model.fit(
    data=bkt_train,
    defaults={
        "order_id": "order_id",
        "student_id": "user_id",
        "skill_name": "skill_id",
        "correct": "correct"
    }
)

print(f"BKT training time: {time.time() - start:.2f} seconds")
print("BKT fitting complete.")

BKT training time: 7.91 seconds
BKT fitting complete.


## Calculate AUCs

In [ ]:
auc_q1 = bkt_model.evaluate(
    data=bkt_q1,
    metric="auc"
)

auc_q2 = bkt_model.evaluate(
    data=bkt_q2,
    metric="auc"
)

auc_q3 = bkt_model.evaluate(
    data=bkt_q3,
    metric="auc"
)

auc_q4 = bkt_model.evaluate(
    data=bkt_q4,
    metric="auc"
)

print(f"Q1 AUC: {auc_q1:.6f}")
print(f"Q2 AUC: {auc_q2:.6f}")
print(f"Q3 AUC: {auc_q3:.6f}")
print(f"Q4 AUC: {auc_q4:.6f}")

Q1 AUC: 0.729531
Q2 AUC: 0.735898
Q3 AUC: 0.713820
Q4 AUC: 0.813358


## Calculate Brier scores

In [ ]:
from sklearn.metrics import brier_score_loss

q1_predictions = bkt_model.predict(data=bkt_q1)
q2_predictions = bkt_model.predict(data=bkt_q2)
q3_predictions = bkt_model.predict(data=bkt_q3)
q4_predictions = bkt_model.predict(data=bkt_q4)

brier_q1 = brier_score_loss(
    q1_predictions["correct"],
    q1_predictions["correct_predictions"]
)

brier_q2 = brier_score_loss(
    q2_predictions["correct"],
    q2_predictions["correct_predictions"]
)

brier_q3 = brier_score_loss(
    q3_predictions["correct"],
    q3_predictions["correct_predictions"]
)

brier_q4 = brier_score_loss(
    q4_predictions["correct"],
    q4_predictions["correct_predictions"]
)

print(f"Q1 Brier Score: {brier_q1:.6f}")
print(f"Q2 Brier Score: {brier_q2:.6f}")
print(f"Q3 Brier Score: {brier_q3:.6f}")
print(f"Q4 Brier Score: {brier_q4:.6f}")

Q1 Brier Score: 0.217788
Q2 Brier Score: 0.204458
Q3 Brier Score: 0.168277
Q4 Brier Score: 0.089306


## Save the data

In [ ]:
bkt_results = pd.DataFrame({
    "ability_quartile": ["Q1", "Q2", "Q3", "Q4"],
    "auc": [
        auc_q1,
        auc_q2,
        auc_q3,
        auc_q4
    ],
    "brier_score": [
        brier_q1,
        brier_q2,
        brier_q3,
        brier_q4
    ]
})

output_path = os.path.join(
    RELIABILITY_DIR,
    "bkt_reliability_results.csv"
)

bkt_results.to_csv(
    output_path,
    index=False
)

print(bkt_results)
print("\nSaved:", output_path)

  ability_quartile     auc  brier_score
0               Q1 0.72953      0.21779
1               Q2 0.73590      0.20446
2               Q3 0.71382      0.16828
3               Q4 0.81336      0.08931

Saved: /content/drive/MyDrive/education-ml-research/ASSISTments2009/reliability_experiment/bkt_reliability_results.csv


## Results

The BKT model was trained once using the training dataset and then evaluated separately on four student ability quartiles. Two complementary metrics were used to evaluate predictive performance: AUC and Brier score. AUC measures how well the model distinguishes between correct and incorrect responses, while the Brier score measures the accuracy of the model's predicted probabilities. Unlike AUC, where higher values indicate better performance, lower Brier scores indicate better probabilistic prediction, with a score of 0 representing perfect predictions.

The model achieved an AUC of 0.7295 for Q1, 0.7359 for Q2, 0.7138 for Q3, and 0.8134 for Q4. Thus, BKT performance varied substantially across student ability groups, with a difference of approximately 0.10 AUC between the lowest-performing group (Q3) and highest-performing group (Q4).

The Brier scores were 0.2178 for Q1, 0.2045 for Q2, 0.1683 for Q3, and 0.0893 for Q4. These results show a clear improvement in probabilistic prediction as student ability increased. The Q4 group had the lowest Brier score (0.0893), indicating that BKT's predicted probabilities were substantially closer to the students' actual outcomes for the highest-ability students. In contrast, Q1 had the highest Brier score (0.2178), indicating greater prediction error for the lowest-ability group.

The Brier score results therefore provide additional evidence of differences in model behavior across ability groups. Unlike the AUC results, which showed a non-monotonic pattern across quartiles, Brier scores decreased consistently from Q1 to Q4. This suggests that although BKT's ability to rank correct versus incorrect responses did not improve consistently with student ability, the accuracy of its predicted probabilities did improve substantially for higher-ability students.

The overall BKT AUC from the  baseline experiment was 0.7945. The quartile AUCs do not need to average to this value because AUC calculated across the entire test set is not equivalent to the average of AUCs calculated separately for subgroups.

Overall, these findings provide evidence that student ability affects the predictive performance of BKT. The substantial variation in both AUC and Brier score across ability groups suggests that evaluating a knowledge tracing model using only an overall performance metric may conceal meaningful differences in how well its predictions perform for different types of students. In particular, the Brier score indicates that BKT's probabilistic predictions were considerably more accurate for higher-ability students than for lower-ability students.